# 🧠 Memory / Persistence — hội thoại đa lượt có nhớ

> Demo nâng cao cho lớp GenAI.

## Ý tưởng

Mặc định mỗi lần `invoke` là độc lập — agent **quên sạch** lượt trước. Để có trí nhớ hội thoại,
LangGraph dùng:

- **Checkpointer** (`InMemorySaver`): tự động lưu lại state sau mỗi bước.
- **`thread_id`**: "mã hội thoại". Cùng `thread_id` = cùng một mạch nhớ; khác `thread_id` = hội thoại mới tinh.
- **Reducer `add_messages`**: gộp tin nhắn mới vào lịch sử thay vì ghi đè.

Ta sẽ thấy: hỏi "đổi thành 7 ngày" ở lượt 2 mà agent vẫn nhớ đang nói về Tokyo.

## 1. Cài đặt & import

In [ ]:
import warnings
# Lọc vài cảnh báo vô hại cho gọn output (kết quả vẫn đúng)
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import os
import json
from typing import TypedDict, Optional, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

from IPython.display import Image, Markdown, display

load_dotenv()

# --- Bổ sung cho demo nâng cao ---
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
print("Imports OK")

## 2. LLM + Langfuse

In [ ]:
# --- Cấu hình ---
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.7

# --- LLM ---
llm = init_chat_model(model=MODEL_NAME, temperature=TEMPERATURE)

# --- Langfuse: phải khởi tạo global client trước, rồi tạo CallbackHandler ---
_host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=_host,
)
langfuse_handler = CallbackHandler()

try:
    if get_client().auth_check():
        print("✅ Langfuse đã kết nối:", _host)
except Exception:
    print("⚠️  Chưa kết nối Langfuse (kiểm tra key/host trong .env). Agent vẫn chạy, chỉ là không có trace.")

print("Model:", MODEL_NAME)

## 3. State hội thoại + node assistant

State chỉ cần một danh sách `messages` với reducer `add_messages` (tự nối tin nhắn mới vào lịch sử).

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]   # add_messages = nối thêm, không ghi đè


SYSTEM = SystemMessage(content=(
    "Bạn là trợ lý du lịch thân thiện, trả lời ngắn gọn bằng tiếng Việt. "
    "Hãy nhớ ngữ cảnh các lượt trước trong cùng cuộc trò chuyện."
))


def assistant(state: ChatState) -> dict:
    # state["messages"] chứa TOÀN BỘ lịch sử (nhờ checkpointer + add_messages)
    reply = llm.invoke([SYSTEM] + state["messages"], config={"callbacks": [langfuse_handler]})
    return {"messages": [reply]}

## 4. Dựng graph + **checkpointer**

Graph cực đơn giản: `START → assistant → END`. Điều "thần kỳ" nằm ở `compile(checkpointer=...)`.

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("assistant", assistant)
graph.add_edge(START, "assistant")
graph.add_edge("assistant", END)

##### TODO: Thực hành #####
# Yêu cầu:
#   - Compile graph và gán vào `app`.
#   - ⭐ Truyền checkpointer=InMemorySaver() vào compile — ĐÂY là thứ tạo ra "bộ nhớ".
#     Không có nó thì mỗi lần invoke là một tờ giấy trắng.
#######################
### START CODE HERE ###
#######################


##### End TODO #####

print("✅ Đã dựng graph hội thoại có nhớ")

## 5. Hội thoại đa lượt trên CÙNG một `thread_id`

Lượt 1 nói về Tokyo. Lượt 2 chỉ nói "đổi thành 7 ngày" — không nhắc Tokyo nữa — xem agent có nhớ không.

In [ ]:
def chat(text, thread_id):
    ##### TODO: Thực hành #####
    # Yêu cầu:
    #   - Tạo config `cfg` chứa thread_id:
    #       {"configurable": {"thread_id": thread_id}, "callbacks": [langfuse_handler]}
    #     thread_id chính là "khoá" để checkpointer biết đây là cuộc hội thoại nào.
    #   - Gọi app.invoke với 1 HumanMessage(content=text) và config=cfg, lưu kết quả vào `out`.
    #######################
    ### START CODE HERE ###
    #######################

    ##### End TODO #####
    print(f"[{thread_id}] 👤 {text}")
    print(f"[{thread_id}] 🤖 {out['messages'][-1].content}\n")
    return out


chat("Tôi muốn đi Tokyo 5 ngày, thích ẩm thực.", "user-A")
chat("Đổi thành 7 ngày nhé.", "user-A")   # không nhắc Tokyo -> agent vẫn phải nhớ

## 6. `thread_id` khác = hội thoại MỚI (không nhớ gì)

Cùng câu hỏi mơ hồ nhưng ở thread khác → agent không biết "nơi đó" là đâu.

In [ ]:
chat("Nơi đó có gì ăn ngon?", "user-B")   # thread mới -> không có ngữ cảnh Tokyo

## 7. Soi bộ nhớ đã lưu

Checkpointer cho phép đọc lại state hiện tại (`get_state`) và toàn bộ lịch sử checkpoint (`get_state_history`).

In [ ]:
cfg_a = {"configurable": {"thread_id": "user-A"}}

snap = app.get_state(cfg_a)
print("Số tin nhắn đã lưu cho user-A:", len(snap.values["messages"]))
for m in snap.values["messages"]:
    print(f"  - {type(m).__name__}: {m.content[:60]}")

print("\nSố checkpoint trong lịch sử user-A:", len(list(app.get_state_history(cfg_a))))

## 8. Tổng kết — Memory / Persistence

- **Checkpointer** tự lưu state sau mỗi bước; **`thread_id`** chia tách các cuộc trò chuyện.
- Cùng `thread_id` → agent nhớ ngữ cảnh; khác `thread_id` → bắt đầu lại từ đầu.
- `add_messages` giúp lịch sử hội thoại được *nối thêm* thay vì bị ghi đè.
- `get_state` / `get_state_history` để soi và "tua lại" trạng thái.

> Demo dùng `InMemorySaver` (mất khi tắt kernel). Sản phẩm thật thường dùng checkpointer bền vững
> (SQLite/Postgres) — cần cài thêm gói, nhưng API sử dụng y hệt.